In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the main dataset
#df = pd.read_csv("ai_dependency_career_anxiety_students.csv")
df = pd.read_csv("ai_student_impact_dataset.csv")

# 2. Load the feature dictionary
#feature_dict = pd.read_csv("feature_dictionary.csv")

# 3. Load metadata
#with open("dataset_metadata.json", "r") as f:
#    metadata = json.load(f)

print("Loaded successfully! Shape:", df.shape)

Loaded successfully! Shape: (50000, 16)


In [5]:
# Set visual styling for publication-ready charts
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

# 1. Load Dataset
df = pd.read_csv("ai_student_impact_dataset.csv")

print("--- Dataset Overview ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
print("\nMissing Values per Column:")
print(df.isnull().sum())
print("\nBasic Summary Statistics:")
print(
    df.describe(
        include=[
            "number",
        ]
    )
)

# 2. Correlation Matrix & Heatmap (Numerical Features)
plt.figure(figsize=(10, 8))
num_cols = [
    "Pre_Semester_GPA",
    "Post_Semester_GPA",
    "Weekly_GenAI_Hours",
    "Traditional_Study_Hours",
    "Perceived_AI_Dependency",
    "Anxiety_Level_During_Exams",
    "Skill_Retention_Score",
]

corr_matrix = df[num_cols].corr()
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1,
)
plt.title("Correlation Matrix of AI Usage, Study Habits, and Performance")
plt.savefig("correlation_heatmap.png", dpi=300)
plt.close()

# 3. Major Disparities Analysis (Categorical vs. Continuous)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Major_Category",
    y="Weekly_GenAI_Hours",
    palette="Set2",
    ax=axes[0],
)
axes[0].set_title("Weekly GenAI Hours by Major Category")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(
    data=df,
    x="Major_Category",
    y="Post_Semester_GPA",
    palette="Set2",
    ax=axes[1],
)
axes[1].set_title("Post-Semester GPA by Major Category")
axes[1].tick_params(axis="x", rotation=30)

plt.savefig("major_disparities.png", dpi=300)
plt.close()

# 4. Non-Linear Relationship / Threshold Analysis
# Exploring Weekly GenAI Hours vs. Post-Semester GPA (with a smoothing trendline)
plt.figure(figsize=(9, 6))
sns.regplot(
    data=df,
    x="Weekly_GenAI_Hours",
    y="Post_Semester_GPA",
    scatter_kws={"alpha": 0.4},
    line_kws={"color": "red"},
    lowess=True,  # LOESS curve to catch non-linear inflection points ("the cliff")
)
plt.title(
    "Non-Linear Threshold Analysis: Weekly GenAI Hours vs. Post-Semester GPA"
)
plt.xlabel("Weekly GenAI Hours")
plt.ylabel("Post-Semester GPA")
plt.savefig("threshold_analysis_gpa.png", dpi=300)
plt.close()

# 5. Retention Trade-off: Dependency vs Skill Retention
plt.figure(figsize=(9, 6))
sns.boxplot(
    data=df, x="Perceived_AI_Dependency", y="Skill_Retention_Score", palette="Blues"
)
plt.title(
    "Cognitive Trade-off: Perceived AI Dependency vs. Skill Retention Score"
)
plt.xlabel("Perceived AI Dependency Level")
plt.ylabel("Skill Retention Score")
plt.savefig("retention_tradeoff.png", dpi=300)
plt.close()

print("\nEDA Complete! All visualization plots saved to disk.")

--- Dataset Overview ---
Total Rows: 50000, Total Columns: 16

Missing Values per Column:
Student_ID                    0
Major_Category                0
Year_of_Study                 0
Pre_Semester_GPA              0
Weekly_GenAI_Hours            0
Primary_Use_Case              0
Prompt_Engineering_Skill      0
Tool_Diversity                0
Paid_Subscription             0
Traditional_Study_Hours       0
Perceived_AI_Dependency       0
Institutional_Policy          0
Anxiety_Level_During_Exams    0
Post_Semester_GPA             0
Skill_Retention_Score         0
Burnout_Risk_Level            0
dtype: int64

Basic Summary Statistics:
          Student_ID  Pre_Semester_GPA  Weekly_GenAI_Hours  Tool_Diversity  \
count   50000.000000      50000.000000        50000.000000     50000.00000   
mean   125000.500000          3.146102            8.427752         2.80026   
std     14433.901067          0.478854            8.269490         1.18802   
min    100001.000000          1.183000        

C:\Users\cecil\AppData\Local\Temp\ipykernel_18052\2731772548.py:50: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\cecil\AppData\Local\Temp\ipykernel_18052\2731772548.py:60: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\cecil\AppData\Local\Temp\ipykernel_18052\2731772548.py:94: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(



EDA Complete! All visualization plots saved to disk.


In [ ]:
# 1. Select and prepare features for clustering
features = [
    "Weekly_GenAI_Hours",
    "Traditional_Study_Hours",
    "Perceived_AI_Dependency",
    "Anxiety_Level_During_Exams",
    "Skill_Retention_Score",
]
X = df[features].dropna()

# 2. Standardize features (crucial for distance-based clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Apply PCA to reduce dimensionality down to 2 components for plotting
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Print explained variance ratio (great to include in your report!)
print(f"Explained Variance by 2 Components: {pca.explained_variance_ratio_.sum():.2f}")

# 5. Fit K-Means Clustering on the scaled data (e.g., 4 clusters)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# 6. Plot the clusters in 2D PCA space
plt.figure(figsize=(9, 6))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=cluster_labels,
    palette="Set1",
    s=70,
    alpha=0.8,
)
plt.title("Student Behavioral Clusters Visualized via PCA (PC1 vs PC2)")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Var)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Var)")
plt.legend(title="Cluster Archetype")
plt.savefig("pca_clusters_visualization.png", dpi=300)
plt.close()

print("PCA complete! Visualization saved as 'pca_clusters_visualization.png'.")


         Weekly_GenAI_Hours  Traditional_Study_Hours  Perceived_AI_Dependency  \
Cluster                                                                         
0                  4.942205                15.418844                 2.657559   
1                 24.522750                 8.963991                 6.387402   
2                  7.854190                11.371260                 3.882985   
3                  5.181138                 7.983679                 2.728818   

         Anxiety_Level_During_Exams  Skill_Retention_Score  
Cluster                                                     
0                          2.969912              82.901689  
1                          5.598995              69.243278  
2                          6.191798              80.912106  
3                          3.296613              67.426597  
